#### Vector Search foundational embedding model

In [0]:
%pip install --upgrade --force-reinstall databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

In [0]:
help(VectorSearchClient)

In [0]:
catalog_name = "end_to_end_retail"
schema_name = "db_landing"
table = "v_db_target"

- Fixed sized chunks do not make sense; better based on regex limits, new line, paragraph, and so on. This way chunk will have meaning. Ideally, chunk should not split a full sentence.

In [0]:
%sql
SELECT customer_id, product_name FROM end_to_end_retail.db_landing.tbl_source_files_sales limit 10

In [0]:
source_df = spark.sql(f"SELECT customer_id, product_name FROM end_to_end_retail.db_landing.tbl_source_files_sales")

- need to spend more time thinking about this chunk approach***

In [0]:
import tiktoken
import pandas as pd

# The GTE model has been trained on a max context lenth of 8192 tokens.
max_chunk_tokens = 20
encoding = tiktoken.get_encoding("cl100k_base")

def chunk_text(text):
    # Encode and then decode within the UDF
    tokens = encoding.encode(text)
    chunks = []
    while tokens:
        chunk_tokens = tokens[:max_chunk_tokens]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)
        tokens = tokens[max_chunk_tokens:]
    return chunks

# Process the data and store in a new list
pandas_df = source_df.toPandas()
processed_data = []
for index, row in pandas_df.iterrows():
    text_chunks = chunk_text(row['product_name'])
    chunk_no = 0
    for chunk in text_chunks:
        row_data = row.to_dict()
        
        # replace the id column with a new unique chunk id
        # and the text column with the text chunk
        row_data['customer_id'] = f"{row['customer_id']}_{chunk_no}"
        row_data['product_name'] = chunk
        
        processed_data.append(row_data)
        chunk_no += 1

chunked_pandas_df = pd.DataFrame(processed_data)
chunked_spark_df = spark.createDataFrame(chunked_pandas_df)

table = "end_to_end_retail.db_landing.v_db_target"

# Write the chunked DataFrame to a Delta table
spark.sql(f"DROP TABLE IF EXISTS {table}")
chunked_spark_df.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(table)

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.v_db_target limit 100

In [0]:
vector_search_endpoint_name = "vector-search-sales-order-endpoint"

vsc.create_endpoint(
    name=vector_search_endpoint_name,
    endpoint_type="STANDARD" # or "STORAGE_OPTIMIZED"
)

vsc.get_endpoint(
  name=vector_search_endpoint_name
)

source_vector_table = "end_to_end_retail.db_landing.v_db_target"

catalog_name = "end_to_end_retail"
schema_name = "db_landing"

# Vector index
vs_index = f"{source_vector_table}_gte_index"
vs_index_fullname = f"{catalog_name}.{schema_name}.{vs_index}"

embedding_model_endpoint = "databricks-gte-large-en"

- to be continued
https://docs.databricks.com/aws/en/notebooks/source/generative-ai/vector-search-foundation-embedding-model-gte-example.html